# FraudGuard AI
## Credit Card Fraud Detection

In this notebook I test a few models for detecting fraudulent transactions.
The main challenge is class imbalance: fraud cases are very rare, so normal accuracy is not enough.

I start with Logistic Regression from the EDA notebook, then compare it with XGBoost from the second notebook.
I also check how changing the prediction threshold affects fraud recall.

## 1. Import Libraries

I use pandas and numpy for data work, matplotlib/seaborn for plots, and scikit-learn/XGBoost for modeling.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## 2. Load the Dataset

The dataset is loaded directly with pandas, like in the original notebooks.

In [ ]:
df = pd.read_csv("/Users/bbaton/Downloads/creditcard.csv")
df.head()

## 3. First Look at the Data

The dataset has PCA-transformed features `V1` to `V28`, plus `Time`, `Amount`, and the target column `Class`.

- `Class = 0` means normal transaction
- `Class = 1` means fraud transaction

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
df["Class"].value_counts()

In [ ]:
fraud_percent = df["Class"].mean() * 100
print(f"Fraud percentage: {fraud_percent:.4f}%")

In [ ]:
class_counts = df["Class"].value_counts()

plt.figure(figsize=(6, 4))
sns.barplot(x=class_counts.index, y=class_counts.values)
plt.title("Normal vs Fraud Transactions")
plt.xlabel("Class")
plt.ylabel("Count")
plt.xticks([0, 1], ["Normal", "Fraud"])
plt.show()

### Note

The data is very imbalanced. Most transactions are normal.
Because of this, accuracy can be misleading.

For example, a model can predict every transaction as normal and still get very high accuracy.
That is why recall, precision, PR-AUC, and the confusion matrix matter more here.

## 4. Simple EDA

I check median values by class and compare a few important-looking features.
These features were also useful in my earlier notebooks.

In [ ]:
df.groupby("Class").median()

In [ ]:
features_to_plot = ["V14", "V17", "V12", "V10", "V7"]

for feature in features_to_plot:
    plt.figure(figsize=(9, 4))
    sns.histplot(df[df["Class"] == 0][feature], bins=50, color="steelblue", label="Normal", stat="density")
    sns.histplot(df[df["Class"] == 1][feature], bins=50, color="tomato", label="Fraud", stat="density")
    plt.title(f"{feature} Distribution by Class")
    plt.legend()
    plt.show()

In [ ]:
correlations = df.corr(numeric_only=True)["Class"].drop("Class")
correlations.abs().sort_values(ascending=False).head(10)

### EDA Note

Some features have much different distributions for fraud and normal transactions.
`V14`, `V17`, `V12`, `V10`, and `V7` look useful, so I will reuse them later for a smaller model.

## 5. Train/Test Split

I split the data into training and testing sets.
I use `stratify=y` because fraud is rare, and I want both sets to keep the same fraud ratio.

In [ ]:
X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train fraud rate:", y_train.mean())
print("Test fraud rate:", y_test.mean())

## 6. Small Evaluation Function

I use one function to print the main metrics for each model.
This keeps the notebook cleaner and makes the models easier to compare.

In [ ]:
def evaluate_model(y_true, fraud_probs, threshold=0.5, model_name="Model"):
    predictions = (fraud_probs >= threshold).astype(int)

    precision, recall, _ = precision_recall_curve(y_true, fraud_probs)
    pr_auc = auc(recall, precision)

    print(model_name)
    print("Threshold:", threshold)
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, predictions))
    print()
    print(classification_report(y_true, predictions, digits=4))
    print("ROC-AUC:", roc_auc_score(y_true, fraud_probs))
    print("PR-AUC:", pr_auc)

    return {
        "model": model_name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_true, fraud_probs),
        "pr_auc": pr_auc,
    }

## 7. Baseline Model: Logistic Regression

I start with Logistic Regression because it is simple and gives a good baseline.

In [ ]:
log_model = LogisticRegression(max_iter=10000)
log_model.fit(X_train, y_train)

log_probs = log_model.predict_proba(X_test)[:, 1]
log_results_05 = evaluate_model(
    y_test,
    log_probs,
    threshold=0.5,
    model_name="Logistic Regression"
)

### Scaling the Data

Logistic Regression can work better when the features are scaled.
I use `StandardScaler` and train the model again.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model_scaled = LogisticRegression(max_iter=10000)
log_model_scaled.fit(X_train_scaled, y_train)

log_scaled_probs = log_model_scaled.predict_proba(X_test_scaled)[:, 1]
log_scaled_results_05 = evaluate_model(
    y_test,
    log_scaled_probs,
    threshold=0.5,
    model_name="Scaled Logistic Regression"
)

## 8. Threshold Tuning

By default, a model predicts fraud when the probability is above `0.5`.
For fraud detection, this can be too strict.

I test lower thresholds to catch more fraud cases.

In [ ]:
thresholds = [0.5, 0.3, 0.2, 0.1, 0.05]
threshold_results = []

for threshold in thresholds:
    result = evaluate_model(
        y_test,
        log_scaled_probs,
        threshold=threshold,
        model_name="Scaled Logistic Regression"
    )
    threshold_results.append(result)
    print("-" * 60)

threshold_results = pd.DataFrame(threshold_results)
threshold_results

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(threshold_results["threshold"], threshold_results["recall"], marker="o", label="Recall")
plt.plot(threshold_results["threshold"], threshold_results["precision"], marker="o", label="Precision")
plt.gca().invert_xaxis()
plt.title("Precision and Recall at Different Thresholds")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.legend()
plt.show()

### Threshold Note

Lower threshold usually gives higher recall, which means the model catches more fraud.
But precision can go down, meaning there will be more false alarms.

## 9. XGBoost Model

Next I train XGBoost.
This model can capture more complex patterns than Logistic Regression.

In [ ]:
xgb_model = XGBClassifier(
    eval_metric="logloss",
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)

xgb_model.fit(X_train, y_train)

xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
xgb_results_05 = evaluate_model(
    y_test,
    xgb_probs,
    threshold=0.5,
    model_name="XGBoost"
)

In [ ]:
xgb_results_02 = evaluate_model(
    y_test,
    xgb_probs,
    threshold=0.2,
    model_name="XGBoost with Lower Threshold"
)

## 10. Feature Importance

XGBoost can show which features it used most.
This does not explain the real-world meaning of the PCA features, but it helps show where the model finds signal.

In [ ]:
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": xgb_model.feature_importances_
}).sort_values("importance", ascending=False)

importance_df.head(10)

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=importance_df.head(10), x="importance", y="feature")
plt.title("Top 10 XGBoost Feature Importances")
plt.show()

## 11. Smaller Model with Selected Features

In the earlier notebook, I used these features:

`V14`, `V7`, `V12`, `V17`, `V10`

Now I train another XGBoost model using only these columns.

In [ ]:
selected_features = ["V14", "V7", "V12", "V17", "V10"]

X_selected = df[selected_features]

X_train_selected, X_test_selected, y_train_selected, y_test_selected = train_test_split(
    X_selected,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

xgb_selected = XGBClassifier(
    eval_metric="logloss",
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)

xgb_selected.fit(X_train_selected, y_train_selected)

xgb_selected_probs = xgb_selected.predict_proba(X_test_selected)[:, 1]
xgb_selected_results = evaluate_model(
    y_test_selected,
    xgb_selected_probs,
    threshold=0.12,
    model_name="XGBoost Selected Features"
)

## 12. Compare Main Results

I compare the main models using the same metrics.
The most important ones for this project are recall and PR-AUC.

In [ ]:
comparison = pd.DataFrame([
    log_results_05,
    log_scaled_results_05,
    threshold_results[threshold_results["threshold"] == 0.1].iloc[0].to_dict(),
    xgb_results_05,
    xgb_results_02,
    xgb_selected_results,
])

comparison.sort_values(by="pr_auc", ascending=False)

In [ ]:
plt.figure(figsize=(8, 5))

for name, probs in {
    "Scaled Logistic Regression": log_scaled_probs,
    "XGBoost": xgb_probs,
    "XGBoost Selected Features": xgb_selected_probs
}.items():
    precision, recall, _ = precision_recall_curve(y_test, probs)
    pr_auc = auc(recall, precision)
    plt.plot(recall, precision, label=f"{name} PR-AUC={pr_auc:.3f}")

plt.title("Precision-Recall Curves")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.show()

## 13. Final Notes

Logistic Regression gave a useful baseline.
Scaling helped make the baseline cleaner.

XGBoost performed better because it can learn more complex patterns.
Lowering the threshold helped catch more fraud, but it also increased false positives.

The selected-feature model is useful because it shows that a few columns carry a lot of signal.

If I continued this project, I would:

- tune XGBoost parameters more carefully
- test more thresholds
- compare false positives and false negatives as business costs
- use cross-validation for more stable results